# Pipeline

## Tokenizer ---> Model --> Post Processing

In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 61.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [11]:
from transformers import pipeline, AutoTokenizer, AutoModel

In [7]:
classifier  = pipeline("sentiment-analysis")
classifier([
    "i've been waiting for Hugging face course my whole life",
    "I hate this so much",

])



No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.998302698135376},
 {'label': 'NEGATIVE', 'score': 0.9995144605636597}]

In [9]:
name_model = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(name_model)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [16]:
raw_input = [
    "i've been waiting for a Hugging face course my whole life.",
    "I hate this so much",

]
input = tokenizer(raw_input, padding  = True, truncation = True, return_tensors = "pt")
print(input)

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662,  2227,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   102,     0,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}


In [12]:
model = AutoModel.from_pretrained(name_model)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.weight     | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
outputs = model(**input)
print(outputs.last_hidden_state.shape)
# outputs["last_hidden_state"]

torch.Size([2, 16, 768])


# Model

There are many different architectures available in 🤗 Transformers, with each one designed around tackling a specific task. Here is a non-exhaustive list: *Model (retrieve the hidden states)

*ForCausalLM

*ForMaskedLM

*ForMultipleChoice

*ForQuestionAnswering

*ForSequenceClassification

*ForTokenClassification

and others 🤗

In [24]:
import torch
from transformers import AutoModelForSequenceClassification


In [22]:
model = AutoModelForSequenceClassification.from_pretrained(name_model)
output = model(**input)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [23]:
print(output.logits.shape)

torch.Size([2, 2])


#  Postprocessing the output

In [26]:
predictions = torch.nn.functional.softmax(output.logits, dim = -1)
print(predictions)

tensor([[1.7051e-03, 9.9829e-01],
        [9.9951e-01, 4.8549e-04]], grad_fn=<SoftmaxBackward0>)


In [27]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}